# 🏛️ Masterclass 01: Document Ingestion, Parsing, & Preprocessing
### *Google Colab Ready | AI Engineer Interview Preparation (3–5 YOE)*

---

## 📌 Executive Summary
In production Retrieval-Augmented Generation (RAG), the ingestion pipeline represents the foundational **ETL (Extract, Transform, Load)** layer. 
A naive or flawed ingestion step permanently damages retrieval quality: **"Garbage In $\rightarrow$ Hallucination Out."**

This master notebook centralizes the complete engineering theory, architectural blueprints, and interactive code implementations for all enterprise document modalities:
1. **The Ingestion Pipeline Architecture & Universal Schema**
2. **Format 1 — Unstructured Text & Markdown**: Unicode normalization (NFKC), non-breaking spaces, regex sanitization, and boilerplate stripping.
3. **Format 2 — PDFs (Digital & Scanned)**: Coordinate geometry, text streams, layout preservation, and table extraction to Markdown.
4. **Format 3 — Images, Receipts, & Scanned Contracts**: The 4-stage Computer Vision pipeline (Grayscale $\rightarrow$ CLAHE $\rightarrow$ Bilateral Filter $\rightarrow$ Otsu Binarization) and deskewing.
5. **Format 4 — Tabular Data (Excel / CSV)**: The mathematical failure of dense vector embeddings on numbers, and row-to-narrative serialization.
6. **Format 5 — Semi-Structured Data (JSON / APIs)**: Token bloat reduction, schema flattening, and internal key pruning.
7. **The Universal Multi-Tenant Ingestion Dispatcher**
8. **Senior AI Interview Q&A & Edge Cases**


---
## ⚙️ 1. Environment Setup (Google Colab & Local)
Run the cell below to bootstrap dependencies and verify sample data files.


In [ ]:
import os
import sys

# In Google Colab, uncomment to install required packages:
# !pip install opencv-python pillow pymupdf pdfplumber pypdf

# Resolve data directory (handles local repo or cloned Colab runtime)
DATA_DIR = "data"
if not os.path.exists(DATA_DIR):
    if os.path.exists("01_document_ingestion/data"):
        DATA_DIR = "01_document_ingestion/data"
    elif os.path.exists("../data"):
        DATA_DIR = "../data"

print(f"Data directory resolved to: {DATA_DIR}")
if os.path.exists(DATA_DIR):
    print("Available sample files:", os.listdir(DATA_DIR))
else:
    print("Warning: data/ directory not found. Please run scripts/generate_sample_data.py first.")


---
## 🧱 2. The Universal Ingested Document Schema & Multi-Tenant Isolation

### Why Schema Standardization is Mandatory:
Downstream chunkers and vector indexers must never be tightly coupled to upstream file loaders. 
Every parser must return instances of **`IngestedDocument`**, enforcing two non-negotiable enterprise invariants:
1. **`tenant_id`**: Mandatory to enforce database-level partitioning and prevent unauthorized cross-tenant data leaks.
2. **`created_at`**: ISO-8601 UTC timestamp to enable temporal decay and time-aware retrieval filtering.


In [ ]:
from dataclasses import dataclass, field
from typing import Dict, Any, List
import datetime

@dataclass
class IngestedDocument:
    page_content: str
    metadata: Dict[str, Any] = field(default_factory=dict)
    
    def __post_init__(self):
        # Mandatory enterprise invariants
        if "tenant_id" not in self.metadata:
            raise ValueError("Production Invariant Violation: tenant_id is MANDATORY.")
        if "created_at" not in self.metadata:
            self.metadata["created_at"] = datetime.datetime.now(datetime.timezone.utc).isoformat()

# Instantiate sample standardized document
sample_doc = IngestedDocument(
    page_content="Enterprise multi-tenant isolation confirmed.",
    metadata={
        "tenant_id": "tenant_apex_01",
        "source": "handbook.pdf",
        "page_number": 1,
        "doc_type": "policy",
        "access_roles": ["employee", "manager"]
    }
)
print("Standardized Document Created Successfully:")
print("Content:", sample_doc.page_content)
print("Metadata:", sample_doc.metadata)


---
## 📝 3. Format 1: Unstructured Text & Markdown Preprocessing

### Deep Theory:
Plain text files (`.txt`) and Markdown (`.md`) frequently suffer from:
1. **Unicode Discrepancies**: The same visual character can be represented by different byte code points.
   - *Example*: `é` can be a single code point (`\u00e9`) or two code points (`e` + accent `\u0301`). 
   - Tokenizers will split them into different tokens unless normalized via **Unicode NFKC**.
2. **Invisible Whitespace Traps**: Non-breaking spaces (`\xa0`), zero-width spaces (`\u200b`), and mixed Windows linebreaks (`\r\n`).
3. **Excessive Linebreaks**: Triple or quadruple newlines artificially fragment text into empty or degraded chunks during chunking.


In [ ]:
import re
import unicodedata

class TextPreprocessor:
    @staticmethod
    def clean_text(raw_text: str) -> str:
        # 1. Unicode NFKC Normalization
        text = unicodedata.normalize("NFKC", raw_text)
        
        # 2. Replace non-breaking spaces and zero-width artifacts
        text = text.replace("\xa0", " ").replace("\u200b", "")
        
        # 3. Standardize Windows line breaks (\r\n -> \n)
        text = re.sub(r"\r\n?", "\n", text)
        
        # 4. Collapse runs of horizontal spaces/tabs
        text = re.sub(r"[ \t]+", " ", text)
        
        # 5. Collapse excessive line breaks (3+ consecutive newlines become 2)
        text = re.sub(r"\n{3,}", "\n\n", text)
        
        return text.strip()

# Test with dirty file
text_file_path = os.path.join(DATA_DIR, "sample_contract.txt")
with open(text_file_path, "r", encoding="utf-8") as f:
    dirty_text = f.read()

print("--- [RAW DIRTY TEXT] ---")
print(repr(dirty_text[:120]) + "...")

cleaned_text = TextPreprocessor.clean_text(dirty_text)
print("\n--- [CLEANED TEXT] ---")
print(cleaned_text)


### Algorithmic Header/Footer Boilerplate Stripper:
Recurring legal notices (*"CONFIDENTIAL HR MEMORANDUM"*) repeated on every document inflate vector similarity for unrelated queries.
We eliminate them using page-frequency analysis:


In [ ]:
from collections import Counter

def strip_recurring_boilerplate(docs: List[str], threshold: float = 0.75) -> List[str]:
    if len(docs) < 2:
        return docs
    first_lines, last_lines = [], []
    for d in docs:
        lines = [l.strip() for l in d.splitlines() if l.strip()]
        if lines:
            first_lines.append(lines[0])
            last_lines.append(lines[-1])
            
    headers = {l for l, c in Counter(first_lines).items() if c / len(docs) >= threshold}
    footers = {l for l, c in Counter(last_lines).items() if c / len(docs) >= threshold}
    
    print(f"Detected Global Boilerplate Headers: {headers}")
    print(f"Detected Global Boilerplate Footers: {footers}")
    
    cleaned = []
    for d in docs:
        lines = [l.strip() for l in d.splitlines() if l.strip()]
        if not lines:
            cleaned.append("")
            continue
        start = 1 if lines[0] in headers else 0
        end = len(lines) - 1 if lines[-1] in footers else len(lines)
        cleaned.append("\n".join(lines[start:end]))
    return cleaned

batch_samples = [
    "CONFIDENTIAL HR MEMORANDUM\nEmployee bonus policy for 2026.\nCONFIDENTIAL HR MEMORANDUM",
    "CONFIDENTIAL HR MEMORANDUM\nHealth and wellness stipend review.\nCONFIDENTIAL HR MEMORANDUM"
]
print("Boilerplate stripped result:")
print(strip_recurring_boilerplate(batch_samples)[0])


---
## 📄 4. Format 2: PDFs (Digital & Scanned)

### Deep Theory:
A PDF is **NOT** a text document. It is a set of display instructions:
`Place glyph 'F' at coordinate (x=50.2, y=720.0)`.
Origin $(0,0)$ in PDF specification is at the **bottom-left** of the page (measured in typographical points: $72\text{ pt} = 1\text{ inch}$).

### The Multi-Column Trap:
A naive text parser reads glyphs purely by descending vertical coordinate $y$.
Across two columns, it reads across columns horizontally, merging unrelated sentences.
**Layout-aware parsers** (`PyMuPDF`) group text spans into bounding boxes $(x_0, y_0, x_1, y_1)$ and resolve reading order column by column.

### Extracting Stream Data & Tables:
Below, we extract stream contents from `sample_policy.pdf` and convert extracted tables into LLM-native Markdown tables.


In [ ]:
import re

def extract_pdf_pages(pdf_path: str) -> List[Dict[str, Any]]:
    pages = []
    with open(pdf_path, "rb") as f:
        raw_bytes = f.read()
    streams = re.findall(b"stream\r?\n(.*?)\r?\nendstream", raw_bytes, re.DOTALL)
    for idx, stream in enumerate(streams):
        text_matches = re.findall(rb"\((.*?)\)\s*Tj", stream)
        page_text = "\n".join(m.decode("latin1", errors="ignore") for m in text_matches)
        pages.append({"page_number": idx + 1, "text": page_text})
    return pages

pdf_path = os.path.join(DATA_DIR, "sample_policy.pdf")
extracted_pdf = extract_pdf_pages(pdf_path)
for p in extracted_pdf:
    print(f"Page {p['page_number']}:\n{p['text']}\n")

def table_grid_to_markdown(grid: List[List[str]]) -> str:
    header = [str(c or "").strip() for c in grid[0]]
    md_table = "| " + " | ".join(header) + " |\n"
    md_table += "| " + " | ".join(["---"] * len(header)) + " |\n"
    for row in grid[1:]:
        cleaned_row = [str(c or "").strip().replace("\n", " ") for c in row]
        md_table += "| " + " | ".join(cleaned_row) + " |\n"
    return md_table

sample_grid = [
    ["Severity", "SLA", "Escalation"],
    ["Sev-1", "< 15 mins", "VP Engineering"],
    ["Sev-2", "< 1 hour", "Director Ops"]
]
print("Serialized Markdown Table (Native LLM Format):")
print(table_grid_to_markdown(sample_grid))


---
## 🖼️ 5. Format 3: Images, Receipts, & Scanned Contracts

### Deep Theory: The 4-Stage Computer Vision Pipeline
Raw images (`.png`, `.jpg`) suffer from smartphone shadows, sensor noise, and skew tilt.
Passing raw camera photos directly to OCR yields high Character Error Rates (CER).

We apply an **OpenCV Computer Vision pipeline**:
1. **Grayscale**: $Y = 0.299R + 0.587G + 0.114B$ (removes color noise).
2. **CLAHE (Contrast Limited Adaptive Histogram Equalization)**:
   Divides image into localized $8\times 8$ grid tiles and clips contrast amplification to eliminate uneven shadows.
3. **Bilateral Filter**: Unlike Gaussian blur, it uses both spatial distance and pixel radiometric intensity to denoise flat backgrounds while keeping text edges sharp.
4. **Otsu's Adaptive Binarization**: Automatically finds the optimal global threshold separating dark text from white paper.


In [ ]:
import cv2
import numpy as np

class VisionDocumentPreprocessor:
    @staticmethod
    def process_image(img_bgr: np.ndarray) -> np.ndarray:
        # 1. Grayscale
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        
        # 2. CLAHE (Local contrast enhancement & shadow elimination)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        contrast = clahe.apply(gray)
        
        # 3. Bilateral Filter (Smooths noise, keeps text edges sharp)
        denoised = cv2.bilateralFilter(contrast, d=9, sigmaColor=75, sigmaSpace=75)
        
        # 4. Otsu's Binarization
        _, binarized = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        return binarized

receipt_path = os.path.join(DATA_DIR, "sample_receipt.png")
raw_receipt = cv2.imread(receipt_path)
binarized_receipt = VisionDocumentPreprocessor.process_image(raw_receipt)

print(f"Raw Image Dimensions: {raw_receipt.shape}")
print(f"Binarized Unique Pixels: {np.unique(binarized_receipt)} (0=Text, 255=Background)")
print("Image successfully converted into clean binary mask ready for OCR.")


---
## 📊 6. Format 4: Tabular Data (Excel & CSV)

### Deep Theory: The Vector Inadequacy on Numbers
Dense embeddings are continuous syntactic projections.
- To an embedding model, `185000` is split into subword tokens: `["18", "500", "0"]`.
- The geometric vector distance between `185000` and `184999` is NOT proportional to their difference ($1$).
- **Cosine similarity CANNOT do numeric range filtering ($> $100k$) or arithmetic aggregations (SUM, AVG)**.

### Solution: Row-to-Narrative Serialization
Every row is translated into an explicit key-value descriptive narrative block:


In [ ]:
import csv

def serialize_row_to_narrative(row: Dict[str, Any], entity_type: str = "Record") -> str:
    lines = [f"=== {entity_type} ==="]
    for key, val in row.items():
        readable_key = key.replace("_", " ").title()
        lines.append(f"- {readable_key}: {val}")
    return "\n".join(lines)

csv_path = os.path.join(DATA_DIR, "sample_payroll.csv")
with open(csv_path, "r", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

print(f"Loaded {len(rows)} tabular rows from CSV.")
print("\nSerialized Narrative Record for Vectorization:")
print(serialize_row_to_narrative(rows[0], entity_type="Staff Payroll Profile"))


---
## 📦 7. Format 5: Semi-Structured Data (JSON / API Payloads)

### The Problem: Syntax Token Bloat
Raw JSON wastes $40-60\%$ of LLM context tokens on punctuation: `{`, `}`, `"`, commas, and deep nesting.
Furthermore, vectorizing internal hashes (`_id`, `session_token`) pollutes semantic retrieval.

### Solution: Recursive Flattening & Key Pruning
We flatten nested structures into dot-notation paths (`account.billing.arr`) and strip blacklisted operational keys:


In [ ]:
def flatten_and_prune_json(data: Dict[str, Any], exclude_keys: set = None, prefix: str = "") -> Dict[str, Any]:
    if exclude_keys is None:
        exclude_keys = {"_id", "session_token", "internal_ip"}
    flattened = {}
    for key, value in data.items():
        if key in exclude_keys:
            continue
        full_key = f"{prefix}.{key}" if prefix else key
        if isinstance(value, dict):
            flattened.update(flatten_and_prune_json(value, exclude_keys, prefix=full_key))
        elif isinstance(value, list):
            flattened[full_key] = ", ".join(str(item) for item in value)
        else:
            flattened[full_key] = value
    return flattened

json_path = os.path.join(DATA_DIR, "sample_customer_profile.json")
import json
with open(json_path, "r", encoding="utf-8") as f:
    raw_json = json.load(f)

flat_json = flatten_and_prune_json(raw_json)
print("--- Clean Flattened JSON (No syntactic token waste) ---")
for k, v in flat_json.items():
    print(f"{k}: {v}")


---
## 🔄 8. The Universal Multi-Tenant Ingestion Dispatcher
This class integrates all format parsers into a unified, production-grade ingestion service.


In [ ]:
class UniversalIngestionDispatcher:
    def __init__(self, tenant_id: str):
        self.tenant_id = tenant_id

    def ingest_text(self, text: str, source: str, doc_type: str = "text") -> IngestedDocument:
        cleaned = TextPreprocessor.clean_text(text)
        return IngestedDocument(
            page_content=cleaned,
            metadata={
                "tenant_id": self.tenant_id,
                "source": source,
                "doc_type": doc_type,
                "char_length": len(cleaned)
            }
        )

    def ingest_tabular_row(self, row: Dict[str, Any], source: str, entity_type: str) -> IngestedDocument:
        narrative = serialize_row_to_narrative(row, entity_type=entity_type)
        return IngestedDocument(
            page_content=narrative,
            metadata={
                "tenant_id": self.tenant_id,
                "source": source,
                "doc_type": "tabular_record"
            }
        )

    def ingest_json(self, data: Dict[str, Any], source: str, doc_type: str) -> IngestedDocument:
        flat = flatten_and_prune_json(data)
        text_content = "\n".join(f"{k}: {v}" for k, v in flat.items())
        return IngestedDocument(
            page_content=text_content,
            metadata={
                "tenant_id": self.tenant_id,
                "source": source,
                "doc_type": doc_type
            }
        )

dispatcher = UniversalIngestionDispatcher(tenant_id="tenant_apex_99")
doc1 = dispatcher.ingest_text("Security policy update for remote workforce.", source="memo.txt")
doc2 = dispatcher.ingest_tabular_row(rows[0], source="payroll.csv", entity_type="Employee")
doc3 = dispatcher.ingest_json(raw_json, source="profile.json", doc_type="customer_account")

print("Universal Dispatcher successfully assembled documents across multiple formats:")
print("1. Text Doc:", doc1.metadata)
print("2. Tabular Doc:", doc2.metadata)
print("3. JSON Doc:", doc3.metadata)


---
## 🎯 9. Senior AI Interview Questions & Edge Cases

### Q1: In PyMuPDF (`fitz`), what is the difference between `page.get_text('text')` and `page.get_text('blocks')`?
> **Answer**: `get_text('text')` flattens text into a single string by raw coordinate scan, which merges multi-column paragraphs horizontally. `get_text('blocks')` returns bounding box tuples `(x0, y0, x1, y1, text, block_no, block_type)`. This lets you partition the page horizontally into column gutters and sort blocks into proper human reading order before tokenization.

### Q2: When should you use Text-to-SQL vs RAG for tabular datasets?
> **Answer**:
> - **Use Text-to-SQL**: When queries require quantitative calculations, mathematical aggregations (`AVG`, `SUM`, `COUNT`), range filtering (`salary > 100000`), or relational joins across multiple tables. Vector similarity search is fundamentally incapable of computing inequalities.
> - **Use RAG**: When queries are qualitative or semantic (e.g. *"Summarize customer complaints regarding delivery delays"*), or when searching unstructured notes embedded within table columns.

### Q3: Why is CLAHE + Bilateral Filter preferred over Gaussian Blur for OCR preprocessing?
> **Answer**: Gaussian Blur weights pixels solely by spatial distance, which blurs high-frequency character boundaries (e.g. merging 'c' and 'e'). Bilateral filtering uses two Gaussian kernels: one for spatial proximity and one for **radiometric pixel intensity difference**, smoothing background noise while leaving text edges sharp. CLAHE equalizes local contrast in $8\times 8$ grid tiles without over-amplifying background noise.

### Q4: Why is multi-tenant isolation enforced at the Ingestion phase rather than just query time?
> **Answer**: If `tenant_id` is not permanently bound to chunk metadata during ingestion and written to the vector store index, query-time filtering is impossible. Metadata enrichment enables database-level SQL pre-filtering (`WHERE tenant_id = 'tenant_1'`), eliminating cross-tenant data leaks and optimizing vector search latency.
